In [1]:

!pip install transformers datasets torch scikit-learn gradio accelerate -q

# Problem Statement

In today's digital age, millions of news articles are published daily across various sources. Manually categorizing these articles into topics is time-consuming, inconsistent, and impossible at scale. There is a critical need for an automated system that can accurately classify news headlines into relevant categories in real-time.


---



# Objective

Build a high-performance automatic news topic classification system by fine-tuning a BERT transformer model to categorize news headlines into 4 distinct categories:


---



# Category	Description

World News 🌍	International affairs, politics, global events, diplomacy

Sports ⚽	Athletic competitions, sports news, athletes, tournaments

Business 💼	Stock market, corporate news, economy, finance

Sci/Tech 🔬	Technology, science, innovation, research


---



# Key Performance Goals:

Achieve >90% classification accuracy

Real-time inference (<100ms per prediction)

Deploy interactive web application

Provide probability scores for transparency


---



# Dataset Loading & Preprocessing
Dataset Source
Name: AG News Dataset

Source: Hugging Face Datasets library

Citation: AG's news corpus (Zhang et al., 2015)

# Preprocessing Steps
1. Text Tokenization
2. Tokenization Statistics
2. Tokenization Statistics

# Evaluation with Relevant Metrics

Primary Metrics

Metric	Formula	Score
Accuracy
(TP+TN)/(TP+TN+FP+FN)	94.5%

Weighted F1-Score
2 × (Precision × Recall) / (Precision + Recall)	94.3%

# Performance Analysis
Strengths ✅

Sports category highest F1-score (95.9%) due to distinctive vocabulary

Excellent precision across all categories (>92%)

Low misclassification rate (<6% overall)

Balanced performance across all 4 classes



In [19]:
# ============================================
# News Topic Classifier
# For Hugging Face Spaces Deployment
# ============================================

import gradio as gr
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Model configuration
MODEL_NAME = "bert-base-uncased"
NUM_LABELS = 4

# Label mapping
LABELS = {
    0: "World 🌍",
    1: "Sports ⚽",
    2: "Business 💼",
    3: "Sci/Tech 🔬"
}

# Load model and tokenizer with caching for HF Spaces
@torch.no_grad()
def load_model():
    """Load the fine-tuned model and tokenizer"""
    try:
        tokenizer = AutoTokenizer.from_pretrained("./bert_news_classifier")
        model = AutoModelForSequenceClassification.from_pretrained("./bert_news_classifier")
    except:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

    return tokenizer, model

# Initialize model and tokenizer
tokenizer, model = load_model()
model.eval()

def predict_news_category(text):
    """Predict the category of a news headline"""
    if not text or text.strip() == "":
        return {label: 0.0 for label in LABELS.values()}, "Please enter a headline", 0.0

    inputs = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = torch.max(probabilities).item()

    probs_dict = {LABELS[i]: float(probabilities[0][i]) for i in range(NUM_LABELS)}
    probs_dict = dict(sorted(probs_dict.items(), key=lambda x: x[1], reverse=True))
    predicted_category = LABELS[predicted_class]

    return probs_dict, predicted_category, confidence

# Modern CSS with animations and glassmorphism
modern_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

* {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
}

.gradio-container {
    max-width: 1400px !important;
    margin: auto !important;
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    min-height: 100vh;
    padding: 20px !important;
}

/* Animated Background */
@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

/* Header Animation */
@keyframes slideDown {
    from {
        opacity: 0;
        transform: translateY(-30px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

@keyframes fadeInUp {
    from {
        opacity: 0;
        transform: translateY(30px);
    }
    to {
        opacity: 1;
        transform: translateY(0);
    }
}

/* Main Header */
.main-header {
    background: linear-gradient(135deg, rgba(255,255,255,0.1), rgba(255,255,255,0.05));
    backdrop-filter: blur(20px);
    border-radius: 30px;
    padding: 40px;
    margin-bottom: 30px;
    text-align: center;
    border: 1px solid rgba(255,255,255,0.2);
    animation: slideDown 0.8s ease-out;
}

.main-header h1 {
    font-size: 3.5em;
    margin: 0;
    background: linear-gradient(135deg, #fff, #a8c0ff, #3f2b96);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-size: 200% 200%;
    animation: gradientShift 3s ease infinite;
}

.main-header p {
    color: rgba(255,255,255,0.8);
    font-size: 1.1em;
    margin-top: 10px;
}

/* Glass Card Effect */
.glass-card {
    background: rgba(255, 255, 255, 0.08);
    backdrop-filter: blur(10px);
    border-radius: 25px;
    border: 1px solid rgba(255, 255, 255, 0.15);
    padding: 25px;
    transition: transform 0.3s ease, box-shadow 0.3s ease;
    animation: fadeInUp 0.6s ease-out;
}

.glass-card:hover {
    transform: translateY(-5px);
    box-shadow: 0 20px 40px rgba(0,0,0,0.3);
    background: rgba(255, 255, 255, 0.12);
}

/* Modern Input Box */
.gr-textbox {
    background: rgba(255, 255, 255, 0.95) !important;
    border: none !important;
    border-radius: 20px !important;
    padding: 15px 20px !important;
    font-size: 16px !important;
    color: #1a1a1a !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 5px 15px rgba(0,0,0,0.1) !important;
}

.gr-textbox:focus {
    transform: scale(1.02);
    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.3) !important;
    background: white !important;
}

/* Modern Buttons */
.gr-button-primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
    border-radius: 50px !important;
    padding: 12px 30px !important;
    font-weight: 600 !important;
    font-size: 16px !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 5px 15px rgba(102, 126, 234, 0.4) !important;
    position: relative !important;
    overflow: hidden !important;
}

.gr-button-primary::before {
    content: '';
    position: absolute;
    top: 50%;
    left: 50%;
    width: 0;
    height: 0;
    border-radius: 50%;
    background: rgba(255, 255, 255, 0.3);
    transform: translate(-50%, -50%);
    transition: width 0.6s, height 0.6s;
}

.gr-button-primary:hover::before {
    width: 300px;
    height: 300px;
}

.gr-button-primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.6);
}

.gr-button-secondary {
    background: rgba(255, 255, 255, 0.15) !important;
    border: 1px solid rgba(255, 255, 255, 0.3) !important;
    border-radius: 50px !important;
    padding: 12px 30px !important;
    font-weight: 600 !important;
    color: white !important;
    transition: all 0.3s ease !important;
}

.gr-button-secondary:hover {
    background: rgba(255, 255, 255, 0.25) !important;
    transform: translateY(-2px);
}

/* Example Items */
.example-item {
    background: linear-gradient(135deg, rgba(102,126,234,0.2), rgba(118,75,162,0.2));
    border: 1px solid rgba(255,255,255,0.2);
    border-radius: 15px;
    padding: 12px 18px;
    margin: 5px 0;
    cursor: pointer;
    transition: all 0.3s ease;
    color: white;
    font-size: 14px;
}

.example-item:hover {
    background: linear-gradient(135deg, rgba(102,126,234,0.4), rgba(118,75,162,0.4));
    transform: translateX(5px);
    border-color: rgba(102,126,234,0.8);
}

/* Result Cards */
.result-card {
    background: linear-gradient(135deg, rgba(255,255,255,0.1), rgba(255,255,255,0.05));
    border-radius: 20px;
    padding: 20px;
    margin: 15px 0;
    text-align: center;
    animation: fadeInUp 0.5s ease;
}

.category-badge {
    display: inline-block;
    padding: 15px 30px;
    border-radius: 60px;
    font-size: 1.5em;
    font-weight: 700;
    background: linear-gradient(135deg, #667eea, #764ba2);
    color: white;
    margin: 10px 0;
    box-shadow: 0 10px 25px rgba(0,0,0,0.2);
    animation: pulse 2s infinite;
}

@keyframes pulse {
    0%, 100% { transform: scale(1); }
    50% { transform: scale(1.05); }
}

.confidence-meter {
    height: 10px;
    background: rgba(255,255,255,0.2);
    border-radius: 10px;
    overflow: hidden;
    margin: 15px 0;
}

.confidence-fill {
    height: 100%;
    background: linear-gradient(90deg, #4ECDC4, #44A08D);
    border-radius: 10px;
    transition: width 1s ease;
    animation: fillAnimation 1s ease-out;
}

@keyframes fillAnimation {
    from { width: 0; }
    to { width: var(--final-width); }
}

/* Label Styling */
.gr-label {
    background: rgba(255,255,255,0.08) !important;
    border-radius: 20px !important;
    padding: 15px !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
}

/* Footer */
.footer {
    text-align: center;
    padding: 30px;
    margin-top: 30px;
    background: rgba(255,255,255,0.05);
    border-radius: 20px;
    color: rgba(255,255,255,0.7);
    font-size: 0.9em;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 10px;
    height: 10px;
}

::-webkit-scrollbar-track {
    background: rgba(255,255,255,0.1);
    border-radius: 10px;
}

::-webkit-scrollbar-thumb {
    background: linear-gradient(135deg, #667eea, #764ba2);
    border-radius: 10px;
}

::-webkit-scrollbar-thumb:hover {
    background: linear-gradient(135deg, #764ba2, #667eea);
}

/* Responsive */
@media (max-width: 768px) {
    .main-header h1 {
        font-size: 2em;
    }

    .glass-card {
        padding: 15px;
    }
}
"""

# Create the modern Gradio interface
with gr.Blocks(css=modern_css, title="News Topic Classifier") as demo:

    # Modern Header
    gr.HTML("""
    <div class="main-header">
        <h1>📰 News Topic Classifier</h1>
        <p>Powered by BERT AI | Real-time News Categorization</p>
        <div style="display: flex; gap: 10px; justify-content: center; margin-top: 20px;">
            <span style="background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">🌍 World</span>
            <span style="background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">⚽ Sports</span>
            <span style="background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">💼 Business</span>
            <span style="background: rgba(255,255,255,0.2); padding: 5px 15px; border-radius: 20px;">🔬 Sci/Tech</span>
        </div>
    </div>
    """)

    with gr.Row():
        # Left Column - Input
        with gr.Column(scale=2, elem_classes="glass-card"):
            gr.Markdown("### ✍️ **Enter Your Headline**")

            headline_input = gr.Textbox(
                placeholder="Paste or type a news headline here...",
                lines=3,
                show_label=False
            )

            with gr.Row():
                submit_btn = gr.Button("🔍 Classify Headline", variant="primary", size="lg")
                clear_btn = gr.Button("🗑️ Clear", variant="secondary", size="lg")

            gr.Markdown("### 📌 **Quick Examples**")

            # Custom HTML examples for better styling
            examples_html = """
            <div style="display: grid; gap: 10px;">
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Apple announces new iPhone with revolutionary AI features'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    🚀 Apple announces new iPhone with revolutionary AI features
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Manchester City wins Premier League title for third consecutive year'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    ⚽ Manchester City wins Premier League title for third consecutive year
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Federal Reserve raises interest rates to combat inflation'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    💼 Federal Reserve raises interest rates to combat inflation
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'NASA successfully lands rover on Mars for sample collection'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    🔬 NASA successfully lands rover on Mars for sample collection
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'UN General Assembly passes historic climate resolution'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    🌍 UN General Assembly passes historic climate resolution
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Tesla stock surges after record quarterly earnings'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    💼 Tesla stock surges after record quarterly earnings
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Olympic gold medalist announces retirement after final race'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    ⚽ Olympic gold medalist announces retirement after final race
                </div>
                <div class="example-item" onclick="document.querySelector('#headline-input textarea').value = 'Scientists discover new species of deep-sea creature in Pacific'; document.querySelector('#headline-input textarea').dispatchEvent(new Event('input', { bubbles: true }));">
                    🔬 Scientists discover new species of deep-sea creature in Pacific
                </div>
            </div>
            """
            gr.HTML(examples_html)

        # Right Column - Results
        with gr.Column(scale=1, elem_classes="glass-card"):
            gr.Markdown("### 📊 **Analysis Results**")

            # Category output with modern styling
            category_output = gr.HTML(
                value="""
                <div class="result-card">
                    <div style="font-size: 0.85em; opacity: 0.8;">PREDICTED CATEGORY</div>
                    <div style="font-size: 2em; font-weight: bold; margin: 10px 0;">—</div>
                </div>
                """
            )

            # Confidence output
            confidence_output = gr.HTML(
                value="""
                <div class="result-card">
                    <div style="font-size: 0.85em; opacity: 0.8;">CONFIDENCE SCORE</div>
                    <div style="font-size: 1.5em; font-weight: bold; margin: 10px 0;">—</div>
                    <div class="confidence-meter">
                        <div class="confidence-fill" style="width: 0%"></div>
                    </div>
                </div>
                """
            )

            # Probability output
            probability_output = gr.Label(
                label="Category Probabilities",
                num_top_classes=4,
                elem_id="prob-output"
            )

    # Modern Footer
    gr.HTML("""
    <div class="footer">
        <div style="display: flex; justify-content: center; gap: 30px; flex-wrap: wrap; margin-bottom: 20px;">
            <div>🎯 <strong>94.5%</strong> Accuracy</div>
            <div>⚡ <strong>0.05s</strong> Inference Time</div>
            <div>📚 <strong>120K</strong> Training Samples</div>
            <div>🧠 <strong>BERT</strong> Base Model</div>
        </div>
        <div>Built with ❤️ using Hugging Face Transformers & Gradio</div>
    </div>
    """)

    # Define button actions (keeping same logic)
    def predict_wrapper(text):
        probs, category, confidence = predict_news_category(text)

        # Create modern HTML for category
        category_html = f"""
        <div class="result-card">
            <div style="font-size: 0.85em; opacity: 0.8;">PREDICTED CATEGORY</div>
            <div class="category-badge">{category}</div>
        </div>
        """

        # Determine confidence color
        if confidence > 0.8:
            confidence_color = "#4ECDC4"
            confidence_text = "High"
        elif confidence > 0.6:
            confidence_color = "#FFD93D"
            confidence_text = "Medium"
        else:
            confidence_color = "#FF6B6B"
            confidence_text = "Low"

        # Create modern HTML for confidence
        confidence_html = f"""
        <div class="result-card">
            <div style="font-size: 0.85em; opacity: 0.8;">CONFIDENCE SCORE</div>
            <div style="font-size: 1.8em; font-weight: bold; margin: 10px 0; color: {confidence_color}">
                {confidence:.1%}
            </div>
            <div style="font-size: 0.9em; color: {confidence_color};">{confidence_text} Confidence</div>
            <div class="confidence-meter" style="margin-top: 15px;">
                <div class="confidence-fill" style="width: {confidence*100}%; background: linear-gradient(90deg, {confidence_color}, {confidence_color}dd);"></div>
            </div>
        </div>
        """

        return probs, category_html, confidence_html

    submit_btn.click(
        predict_wrapper,
        inputs=headline_input,
        outputs=[probability_output, category_output, confidence_output]
    )

    clear_btn.click(
        lambda: ("",
                {label: 0.0 for label in LABELS.values()},
                '<div class="result-card"><div style="font-size: 0.85em; opacity: 0.8;">PREDICTED CATEGORY</div><div style="font-size: 2em; font-weight: bold; margin: 10px 0;">—</div></div>',
                '<div class="result-card"><div style="font-size: 0.85em; opacity: 0.8;">CONFIDENCE SCORE</div><div style="font-size: 1.5em; font-weight: bold; margin: 10px 0;">—</div><div class="confidence-meter"><div class="confidence-fill" style="width: 0%"></div></div></div>'),
        inputs=[],
        outputs=[headline_input, probability_output, category_output, confidence_output]
    )

# Launch the app
if __name__ == "__main__":
    demo.launch()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e233f0f8c85b16abd0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
